# Aircraft classification and caption reliability

Correct VGG16 preprocessing and complete evaluation batches; separate pretrained BLIP descriptions. Caption-quality review remains manual and unmeasured.

New project code prepared with AI assistance from concepts studied in the [IBM AI Engineering program](https://www.coursera.org/professional-certificates/ai-engineer). This is a separate experiment, not an IBM assessment submission. Data/model ownership remains with the original providers; data and weights are not bundled.

A local pilot is not Kaggle execution evidence or proof of a quality improvement.

## Configuration and dependencies

Attach the intended data and cache model weights, or explicitly enable downloads with internet access. Use the project's documented dependency environment. All training defaults to the bounded pilot; set `FULL_RUN=True` only for a separately budgeted complete run. `Save & Run All` must pass in the chosen Kaggle environment before publication.

In [1]:
"""Shared provenance and evaluation helpers for the selected portfolio projects."""

from __future__ import annotations

import hashlib
import importlib.metadata
import json
import platform
import random
from collections import Counter, defaultdict
from datetime import UTC, datetime
from pathlib import Path


def file_sha256(path: Path) -> str:
    """Hash an input/checkpoint without reading the entire file into memory."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def write_json(path: Path, value: dict | list) -> None:
    """Publish a complete JSON artifact atomically."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8"
    )
    temporary.replace(path)


def runtime(packages: list[str]) -> dict:
    """Capture versions without collecting environment variables or host paths."""
    return {
        "utc": datetime.now(UTC).isoformat(),
        "python": platform.python_version(),
        "platform": platform.system(),
        "packages": {name: importlib.metadata.version(name) for name in packages},
    }


def source_identity(project_file: str | None, embedded_digest: str | None = None) -> dict:
    """Bind module runs and generated notebooks to their authored source files."""
    project_digest = file_sha256(Path(project_file)) if project_file else embedded_digest
    common_digest = (
        file_sha256(Path(__file__))
        if "__file__" in globals()
        else globals().get("COMMON_SOURCE_SHA256")
    )
    if not project_digest or not common_digest:
        raise ValueError("Project source identity is missing.")
    return {"project_sha256": project_digest, "common_sha256": common_digest}


def index_images(
    root: Path, class_names: list[str], directories: dict[str, str] | None = None
) -> list[dict]:
    """Index decoded RGB content as well as bytes, retaining only relative filenames."""
    from PIL import Image, ImageOps

    rows = []
    directories = directories or {"unsplit": "."}
    for role, directory in directories.items():
        for label, name in enumerate(class_names):
            folder = root / directory / name
            if not folder.is_dir():
                raise ValueError(f"Missing image class directory: {directory}/{name}")
            for path in sorted(folder.rglob("*")):
                if path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue
                with Image.open(path) as original:
                    image = ImageOps.exif_transpose(original).convert("RGB")
                    pixels = str(image.size).encode() + image.tobytes()
                rows.append(
                    {
                        "file": path.relative_to(root).as_posix(),
                        "label": label,
                        "role": role,
                        "file_sha256": file_sha256(path),
                        "pixel_sha256": hashlib.sha256(pixels).hexdigest(),
                    }
                )
    if not rows:
        raise ValueError("No images found.")
    return rows


def clean_image_splits(rows: list[dict]) -> tuple[list[dict], list[dict]]:
    """Remove exact duplicates and conflicting labels, preserving held-out copies first."""
    groups = defaultdict(list)
    for row in rows:
        groups[row["pixel_sha256"]].append(row)
    retained, removed = [], []
    priority = {"test": 0, "validation": 1, "train": 2}
    for group in groups.values():
        if len({row["label"] for row in group}) > 1:
            removed.extend(
                {"file": row["file"], "reason": "conflicting_pixel_labels"} for row in group
            )
            continue
        ordered = sorted(group, key=lambda row: (priority[row["role"]], row["file"]))
        retained.append(ordered[0])
        removed.extend(
            {"file": row["file"], "reason": "exact_pixel_duplicate"} for row in ordered[1:]
        )
    return sorted(retained, key=lambda row: row["file"]), removed


def balanced_subset(rows: list[dict], per_class: int | None, seed: int) -> list[dict]:
    """Select a reproducible pilot without changing the complete split manifest."""
    if per_class is None:
        return rows
    rng = random.Random(seed)
    result = []
    for label in sorted({row["label"] for row in rows}):
        group = sorted((row for row in rows if row["label"] == label), key=lambda row: row["file"])
        rng.shuffle(group)
        result.extend(group[:per_class])
    return sorted(result, key=lambda row: row["file"])


def split_counts(rows: list[dict]) -> dict:
    """Describe the actual records, including class support."""
    return {
        role: dict(Counter(str(row["label"]) for row in rows if row["role"] == role))
        for role in ["train", "validation", "test"]
    }


def classification_metrics(labels: list[int], probabilities: list[float]) -> dict:
    """Evaluate exactly the predictions supplied, never the size of an upstream dataset."""
    import numpy as np
    from sklearn.metrics import (
        accuracy_score,
        classification_report,
        confusion_matrix,
        f1_score,
        roc_auc_score,
    )

    if not labels or len(labels) != len(probabilities):
        raise ValueError("Predictions and labels must have the same nonzero length.")
    probabilities = np.asarray(probabilities)
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Expected finite probabilities in [0, 1].")
    predicted = (probabilities >= 0.5).astype(int)
    return {
        "samples": len(labels),
        "accuracy": float(accuracy_score(labels, predicted)),
        "macro_f1": float(
            f1_score(labels, predicted, labels=[0, 1], average="macro", zero_division=0)
        ),
        "roc_auc": float(roc_auc_score(labels, probabilities)) if len(set(labels)) == 2 else None,
        "confusion_matrix": confusion_matrix(labels, predicted, labels=[0, 1]).tolist(),
        "per_class": classification_report(
            labels, predicted, labels=[0, 1], output_dict=True, zero_division=0
        ),
    }

COMMON_SOURCE_SHA256 = 'ded7059d91e2123ef7bc9be2823a1d220d905041f45b9dbe83bac59d27547853'
PROJECT_SOURCE_SHA256 = '3d442cb44a949540bd46098c8861b9a1f97e7bcc3bd7a758a72a9fd0593b7426'

"""Train a VGG16 feature classifier and inspect separate BLIP descriptions."""


import argparse
import hashlib
import os
import time
from pathlib import Path

import numpy as np


VGG_NOTOP_MD5 = "6d6bbae143d832006294945121d1f1fc"


def vgg_weight_source(weights: Path, allow_downloads: bool) -> str:
    """Use a verified explicit file path so offline loading cannot invoke get_file."""
    if not weights.is_file():
        if allow_downloads:
            return "imagenet"
        raise FileNotFoundError("Cache the ImageNet VGG16 weights or explicitly allow downloads.")
    digest = hashlib.md5(usedforsecurity=False)
    with weights.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != VGG_NOTOP_MD5:
        raise ValueError("Cached VGG16 weights do not match the published Keras checksum.")
    return str(weights)


def rgb_array(path: Path, size: tuple[int, int] = (224, 224)) -> np.ndarray:
    """Return displayable, unscaled RGB; preprocessing is a separate model operation."""
    from PIL import Image, ImageOps

    with Image.open(path) as image:
        return np.asarray(
            ImageOps.exif_transpose(image).convert("RGB").resize(size), dtype=np.float32
        )


def preprocess_rgb(images: np.ndarray) -> np.ndarray:
    """Apply the pretrained VGG16 contract while preserving RGB display inputs."""
    from tensorflow.keras.applications.vgg16 import preprocess_input

    return preprocess_input(np.asarray(images, dtype=np.float32).copy())


def extract_features(
    root: Path, rows: list[dict], backbone, preprocess, batch_size: int = 8
) -> np.ndarray:
    """Include the final incomplete batch, preserving record order."""
    features = []
    for start in range(0, len(rows), batch_size):
        batch = np.stack(
            [rgb_array(root / row["file"]) for row in rows[start : start + batch_size]]
        )
        features.append(np.asarray(backbone(preprocess(batch.copy()), training=False)))
    if not features:
        raise ValueError("Cannot extract an empty split.")
    return np.concatenate(features)


def caption_image(processor, model, image, max_new_tokens: int = 32) -> str:
    """Run the captioner directly; failures propagate instead of becoming fake captions."""
    import torch

    model.eval()
    inputs = processor(images=image, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {key: value.to(device) for key, value in inputs.items()}
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    caption = processor.decode(output[0], skip_special_tokens=True).strip()
    if not caption:
        raise ValueError("Captioner returned an empty description.")
    return caption


def run(
    data_root: Path,
    output: Path,
    *,
    full: bool = False,
    allow_downloads: bool = False,
    seed: int = 42,
    epochs: int = 10,
) -> dict:
    """Use the existing source splits after exact-pixel leakage/label checks."""
    os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
    os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "1")
    import tensorflow as tf
    from PIL import Image, ImageOps
    from transformers import BlipForConditionalGeneration, BlipProcessor

    tf.keras.utils.set_random_seed(seed)
    rows = index_images(
        data_root, ["crack", "dent"], {"train": "train", "validation": "valid", "test": "test"}
    )
    rows, removed = clean_image_splits(rows)
    write_json(output / "split-manifest.json", {"records": rows, "removed": removed})
    groups = {
        role: [row for row in rows if row["role"] == role]
        for role in ["train", "validation", "test"]
    }
    if any({r["label"] for r in group} != {0, 1} for group in groups.values()):
        raise ValueError("Each cleaned split must contain both crack and dent images.")
    selected = {
        role: balanced_subset(group, None if full else (8 if role == "train" else 4), seed)
        for role, group in groups.items()
    }
    weights = (
        Path(os.environ.get("KERAS_HOME", str(Path.home() / ".keras")))
        / "models/vgg16_weights_tf_dim_ordering_tf_kernels_notop.h5"
    )
    weight_source = vgg_weight_source(weights, allow_downloads)
    started = time.perf_counter()
    backbone = tf.keras.applications.VGG16(include_top=False, weights=weight_source, pooling="avg")
    backbone.trainable = False
    feature_arrays = {
        role: extract_features(data_root, group, backbone, preprocess_rgb)
        for role, group in selected.items()
    }
    labels = {
        role: np.asarray([r["label"] for r in group], dtype=np.float32)
        for role, group in selected.items()
    }
    head = tf.keras.Sequential(
        [
            tf.keras.layers.Input((512,)),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(1),
        ]
    )
    head.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    )
    history, best_loss = [], float("inf")
    checkpoint = output / "best-classifier.keras"
    rng = np.random.default_rng(seed)
    for epoch in range(epochs if full else 2):
        order = rng.permutation(len(selected["train"]))
        for start in range(0, len(order), 8):
            indices = order[start : start + 8]
            head.train_on_batch(feature_arrays["train"][indices], labels["train"][indices, None])
        logits = head(feature_arrays["validation"], training=False)
        loss = float(
            tf.keras.losses.binary_crossentropy(
                labels["validation"][:, None], logits, from_logits=True
            )
            .numpy()
            .mean()
        )
        history.append({"epoch": epoch + 1, "validation_loss": loss})
        if loss < best_loss:
            best_loss = loss
            head.save(checkpoint)
    head = tf.keras.models.load_model(checkpoint)
    expected = np.asarray(head(feature_arrays["validation"], training=False))
    reloaded = tf.keras.models.load_model(checkpoint)
    np.testing.assert_array_equal(
        expected, np.asarray(reloaded(feature_arrays["validation"], training=False))
    )
    probabilities = (
        tf.sigmoid(head(feature_arrays["test"], training=False)).numpy().ravel().tolist()
    )
    metrics = classification_metrics(labels["test"].astype(int).tolist(), probabilities)
    model_id = "Salesforce/blip-image-captioning-base"
    processor = BlipProcessor.from_pretrained(model_id, local_files_only=not allow_downloads)
    captioner = BlipForConditionalGeneration.from_pretrained(
        model_id, local_files_only=not allow_downloads
    )
    captions = []
    for label in [0, 1]:
        index = next(i for i, row in enumerate(selected["test"]) if row["label"] == label)
        row = selected["test"][index]
        with Image.open(data_root / row["file"]) as image:
            text = caption_image(
                processor, captioner, ImageOps.exif_transpose(image).convert("RGB")
            )
        captions.append(
            {
                "file": row["file"],
                "label": ["crack", "dent"][label],
                "classifier_prediction": ["crack", "dent"][int(probabilities[index] >= 0.5)],
                "caption": text,
                "visible_defect_supported": None,
                "hallucination_review": None,
            }
        )
    report = {
        "project": "aircraft",
        "profile": "full" if full else "bounded-pilot",
        "seed": seed,
        "class_names": ["crack", "dent"],
        "complete_split_counts": split_counts(rows),
        "selected_split_counts": split_counts([r for group in selected.values() for r in group]),
        "split_manifest_sha256": file_sha256(output / "split-manifest.json"),
        "vgg_weights_sha256": file_sha256(weights),
        "preprocessing": "RGB 0..255 -> VGG BGR channel centering; no 1/255 rescale",
        "checkpoint_sha256": file_sha256(checkpoint),
        "exact_reload": True,
        "test": {**metrics, "ids": [r["file"] for r in selected["test"]]},
        "history": history,
        "caption_model": model_id,
        "caption_revision": getattr(captioner.config, "_commit_hash", None),
        "captions": captions,
        "caption_quality": "Not evaluated; human review fields remain null.",
        "near_duplicate_or_source_group_leakage_audited": False,
        "seconds": time.perf_counter() - started,
        "source": source_identity(
            globals().get("__file__"), globals().get("PROJECT_SOURCE_SHA256")
        ),
        "runtime": runtime(["tensorflow", "transformers", "torch", "numpy", "Pillow"]),
    }
    write_json(output / "metrics.json", report)
    return report


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--data-root", type=Path, required=True)
    parser.add_argument(
        "--output", type=Path, default=Path(".lab-cache/selected-projects/aircraft")
    )
    parser.add_argument("--full", action="store_true")
    parser.add_argument("--allow-downloads", action="store_true")
    args = parser.parse_args()
    report = run(args.data_root, args.output, full=args.full, allow_downloads=args.allow_downloads)
    print(report["test"])



In [2]:
# Keep this False for the small correctness pilot. Full training requires a separate run.
FULL_RUN = False
ALLOW_DOWNLOADS = False
SEED = 42

# Set this explicitly if multiple attached datasets match.
DATA_PATH = None
if DATA_PATH is None:
    if Path('/kaggle/input').is_dir():
        matches = sorted(Path('/kaggle/input').rglob('aircraft_damage_dataset_v1'))
        if len(matches) != 1:
            raise ValueError('Set DATA_PATH to the intended attached dataset; expected one matching input.')
        DATA_PATH = matches[0]
    else:
        root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'provenance/source-manifest.json').is_file()), None)
        if root is None:
            raise ValueError('Set DATA_PATH to the local data file/directory.')
        DATA_PATH = root / '.lab-cache/02-deep-learning-with-keras/aircraft-damage-classification-and-captioning/aircraft_damage_dataset_v1'
DATA_PATH = Path(DATA_PATH)
root = locals().get('root') or Path.cwd()
OUTPUT = (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else root / '.lab-cache/selected-projects') / 'aircraft'


In [3]:
report = run(DATA_PATH, OUTPUT, full=FULL_RUN, seed=SEED, allow_downloads=ALLOW_DOWNLOADS)
print('Profile:', report['profile'])
print('Actual evaluated split sizes:', report['selected_split_counts'])

Matplotlib is building the font cache; this may take a moment.


<repository>/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Profile: bounded-pilot
Actual evaluated split sizes: {'train': {'0': 8, '1': 8}, 'validation': {'0': 4, '1': 4}, 'test': {'0': 4, '1': 4}}


In [4]:
print('Classifier:', report['test'])
print('Caption quality:', report['caption_quality'])
for row in report['captions']:
    print(row)

Classifier: {'samples': 8, 'accuracy': 0.375, 'macro_f1': 0.36507936507936506, 'roc_auc': 0.4375, 'confusion_matrix': [[1, 3], [2, 2]], 'per_class': {'0': {'precision': 0.3333333333333333, 'recall': 0.25, 'f1-score': 0.2857142857142857, 'support': 4.0}, '1': {'precision': 0.4, 'recall': 0.5, 'f1-score': 0.4444444444444444, 'support': 4.0}, 'accuracy': 0.375, 'macro avg': {'precision': 0.3666666666666667, 'recall': 0.375, 'f1-score': 0.36507936507936506, 'support': 8.0}, 'weighted avg': {'precision': 0.3666666666666667, 'recall': 0.375, 'f1-score': 0.36507936507936506, 'support': 8.0}}, 'ids': ['test/crack/140_1_JPG_jpg.rf.783ec2a489ddc3692ef15efcb14436c6.jpg', 'test/crack/145_4_JPG_jpg.rf.c24479fe868d62e5efe987e248c2b1af.jpg', 'test/crack/14_23_JPG_jpg.rf.b7dbf66f2939f210dcf0bcbd1d814e8f.jpg', 'test/crack/158_24_JPG_jpg.rf.548c65fc199c9888ceb4164fbac1e785.jpg', 'test/dent/105_3_JPG_jpg.rf.f410c0d6b274a5c73623651c30e32e35.jpg', 'test/dent/136_20_JPG_jpg.rf.e13ff88f88a3dbf20345eae5eeeb6b

## Interpretation

Inspect sample counts, split provenance and reload evidence before reading model scores. Pilot scores are too small to rank models or claim robust improvement. Full training, near-duplicate/source overlap review and task-specific quality evaluation remain separate checks.